# 02 Data checks

Read-only counts that decide what the late label means: final stops (3a), duplicates (3b), extreme delays (3c), planned hours per service and station (6), cancellations (3f, 12). These run on July. Task 13 (month boundaries) runs on all six files, loading only key and planned-time columns, never delay or cancellation columns. Nothing here changes the saved outputs of notebook 01.

In [19]:
import numpy as np
import pandas as pd

from dbdelay.config import DELAY_COLUMN, LATE_THRESHOLD_MIN, MONTHS, PLANNED_TIME_COLUMNS
from dbdelay.data import month_path
from dbdelay.evaluation import planned_day

In [20]:
CHECK_MONTH = "2026-07"   # the month these checks run on (shared settings live in src/dbdelay/config.py)

RIDE_KEY = ["train_line_ride_id", "train_line_station_num"]        # one stop of one ride
TIMETABLE_KEY = ["eva", "train_number", "departure_planned_time"]  # same train, station, planned time
SERVICE_KEY = ["train_type", "train_number"]                       # a recurring service, e.g. ICE 123
CANCEL_COLUMNS = ["arrival_is_canceled", "departure_is_canceled"]
EXTREME_DELAY_MIN = 180   # PROPOSAL, not a decision: |delay| above this counts as extreme
EXAMPLE_ROWS = 6          # how many example rows to show

CHECK_COLUMNS = [
    "eva", "train_type", "train_number", DELAY_COLUMN,
    *RIDE_KEY,
    "arrival_planned_time", "arrival_change_time",
    "departure_planned_time", "departure_change_time",
    *CANCEL_COLUMNS,
]

# Task 13 reads every month, September included, so it loads ONLY key and planned-time
# columns: no delay, no cancellation flags (September is the test month).
BOUNDARY_COLUMNS = ["eva", "train_number", *PLANNED_TIME_COLUMNS]

In [21]:
df = pd.read_parquet(month_path(CHECK_MONTH), columns=CHECK_COLUMNS)
print("Month:", CHECK_MONTH)
print("Rows and columns:", df.shape)   # July must show 14,768,104 rows and 12 columns

Month: 2026-07
Rows and columns: (14768104, 12)


In [22]:
def share(part: int, whole: int) -> str:
    """Format a count as 'part of whole (x.x%)'."""
    percent = f"{part / whole:.1%}" if whole else "n/a"
    return f"{part:,} of {whole:,} ({percent})"


def minutes_between(later: pd.Series, earlier: pd.Series) -> pd.Series:
    """Difference between two time columns, in minutes."""
    return (later - earlier).dt.total_seconds() / 60


def delay_match(stops: pd.DataFrame, rows: pd.Series, change_col: str, planned_col: str) -> None:
    """Print how often delay_in_min equals change time minus planned time on the chosen rows.

    Rows missing either time are counted separately, never treated as a mismatch.
    """
    subset = stops.loc[rows, [DELAY_COLUMN, change_col, planned_col]]
    has_times = subset[change_col].notna() & subset[planned_col].notna()
    with_times = subset[has_times]
    computed = minutes_between(with_times[change_col], with_times[planned_col])
    matches = int((computed == with_times[DELAY_COLUMN]).sum())
    print("  Missing a time (not compared):", share(int((~has_times).sum()), len(subset)))
    print(f"  Delay = {change_col} - {planned_col}:", share(matches, len(with_times)))


def month_position(day: pd.Series, month: str) -> pd.Series:
    """Label each planned day 'earlier', 'this month', 'later' or 'missing', relative to the file's month."""
    planned_month = day.dt.to_period("M")
    file_month = pd.Period(month, freq="M")
    labels = pd.Series("this month", index=day.index)
    labels[planned_month < file_month] = "earlier"
    labels[planned_month > file_month] = "later"
    labels[day.isna()] = "missing"
    return labels


def key_fingerprints(stops: pd.DataFrame) -> np.ndarray:
    """One 64-bit fingerprint per stop with a planned departure, built from the timetable key.

    Comparing fingerprints instead of three columns keeps only one month in memory at a time.
    """
    with_departure = stops[stops["departure_planned_time"].notna()]
    return pd.util.hash_pandas_object(with_departure[TIMETABLE_KEY], index=False).to_numpy()

## 3a Final stops

A ride's last stop has no departure. Which delay does `delay_in_min` hold there? We compare it with arrival change minus arrival planned. As a control, the same comparison for departures on normal rows.

In [23]:
no_departure = df["departure_planned_time"].isna()
has_delay = df[DELAY_COLUMN].notna()
last_stop_num = df.groupby("train_line_ride_id")["train_line_station_num"].transform("max")
is_last_stop = df["train_line_station_num"] == last_stop_num

n_no_departure = int(no_departure.sum())
print("Rows with no planned departure:", share(n_no_departure, len(df)))
print("  ...that are the last stop of their ride:", share(int((no_departure & is_last_stop).sum()), n_no_departure))
print("  ...that have a delay value:             ", share(int((no_departure & has_delay).sum()), n_no_departure))
print()
print(pd.crosstab(no_departure.rename("no planned departure"),
                  has_delay.rename("delay present"), margins=True))

print("\nRows with no planned departure but a delay: is it the ARRIVAL delay?")
delay_match(df, no_departure & has_delay, "arrival_change_time", "arrival_planned_time")

print("\nRows with a planned departure and a delay: is it the DEPARTURE delay?")
delay_match(df, ~no_departure & has_delay, "departure_change_time", "departure_planned_time")

Rows with no planned departure: 1,618,335 of 14,768,104 (11.0%)
  ...that are the last stop of their ride: 964,047 of 1,618,335 (59.6%)
  ...that have a delay value:              1,126,514 of 1,618,335 (69.6%)

delay present          False      True       All
no planned departure                            
False                      0  13149769  13149769
True                  491821   1126514   1618335
All                   491821  14276283  14768104

Rows with no planned departure but a delay: is it the ARRIVAL delay?
  Missing a time (not compared): 0 of 1,126,514 (0.0%)
  Delay = arrival_change_time - arrival_planned_time: 1,126,514 of 1,126,514 (100.0%)

Rows with a planned departure and a delay: is it the DEPARTURE delay?
  Missing a time (not compared): 0 of 13,149,769 (0.0%)
  Delay = departure_change_time - departure_planned_time: 13,149,769 of 13,149,769 (100.0%)


## 3b Duplicates by key

"Duplicate rows: 0" in notebook 01 only finds rows that are identical in every column. Here we look for the same stop recorded twice: by ride key (ride id + stop number) and by timetable key (station + train number + planned departure).

In [24]:
# duplicated() marks EXTRA copies: a group of 3 identical keys counts as 2.
ride_dupes = df.duplicated(subset=RIDE_KEY)
print("Extra copies by ride key:     ", share(int(ride_dupes.sum()), len(df)))

# Rows without a planned departure all share NaT, so the same train at the same
# station on different days would look like duplicates. Only rows with a departure are checked.
has_departure = df["departure_planned_time"].notna()
timetable_dupes = df.loc[has_departure, TIMETABLE_KEY].duplicated()
print("Extra copies by timetable key:", share(int(timetable_dupes.sum()), int(has_departure.sum())))

in_ride_group = df.duplicated(subset=RIDE_KEY, keep=False)   # every row of a duplicate group
print("\nExample duplicate rows (ride key):")
display(df[in_ride_group].sort_values(RIDE_KEY).head(EXAMPLE_ROWS))

in_timetable_group = has_departure & df.duplicated(subset=TIMETABLE_KEY, keep=False)
print("Example duplicate rows (timetable key):")
display(df[in_timetable_group].sort_values(TIMETABLE_KEY).head(EXAMPLE_ROWS))

Extra copies by ride key:      13,676,228 of 14,768,104 (92.6%)
Extra copies by timetable key: 1,255 of 13,149,769 (0.0%)

Example duplicate rows (ride key):


,eva,train_type,train_number,delay_in_min,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,arrival_is_canceled,departure_is_canceled
65743,08010133,IC,2456,0.0,-1000672545375930240,1,NaT,NaT,2026-07-01 06:38:00,2026-07-01 06:38:00,False,True
597051,08010133,IC,2456,0.0,-1000672545375930240,1,NaT,NaT,2026-07-02 06:38:00,2026-07-02 06:38:00,False,True
70235,08011956,IC,2456,0.0,-1000672545375930240,2,2026-07-01 06:46:00,2026-07-01 06:46:00,2026-07-01 06:47:00,2026-07-01 06:47:00,True,True
601760,08011956,IC,2456,0.0,-1000672545375930240,2,2026-07-02 06:46:00,2026-07-02 06:46:00,2026-07-02 06:47:00,2026-07-02 06:47:00,True,True
79824,08010366,IC,2456,1.0,-1000672545375930240,3,2026-07-01 07:02:00,2026-07-01 07:02:00,2026-07-01 07:04:00,2026-07-01 07:05:00,True,False
610361,08010366,IC,2456,0.0,-1000672545375930240,3,2026-07-02 07:02:00,2026-07-02 07:02:00,2026-07-02 07:04:00,2026-07-02 07:04:00,True,True


Example duplicate rows (timetable key):


,eva,train_type,train_number,delay_in_min,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,arrival_is_canceled,departure_is_canceled
11876245,08000031,RE,4684,0.0,3924581614387547673,6,2026-07-25 22:59:00,2026-07-25 22:59:00,2026-07-25 22:59:00,2026-07-25 22:59:00,False,False
11877671,08000031,RE,4684,5.0,-6707776539338401670,104,2026-07-25 22:59:00,2026-07-25 23:03:00,2026-07-25 22:59:00,2026-07-25 23:04:00,False,False
4569451,08000032,RB,58019,0.0,-748086963037885863,21,2026-07-10 09:37:00,2026-07-10 09:37:00,2026-07-10 09:38:00,2026-07-10 09:38:00,False,False
4569666,08000032,RB,58019,0.0,-8204721622065213175,119,2026-07-10 09:37:00,2026-07-10 09:37:00,2026-07-10 09:38:00,2026-07-10 09:38:00,False,False
5903235,08000032,RB,58019,0.0,-8204721622065213175,119,2026-07-13 09:37:00,2026-07-13 09:37:00,2026-07-13 09:38:00,2026-07-13 09:38:00,True,True
5903450,08000032,RB,58019,0.0,-748086963037885863,21,2026-07-13 09:37:00,2026-07-13 09:37:00,2026-07-13 09:38:00,2026-07-13 09:38:00,False,False


## 3c Extreme delays (count only)

Nothing is excluded here. Whether to exclude, and at what threshold, is an owner decision after seeing these counts.

In [25]:
delay = df[DELAY_COLUMN]
too_late = delay > EXTREME_DELAY_MIN
too_early = delay < -EXTREME_DELAY_MIN
extreme = too_late | too_early

print(f"Threshold (proposal): |delay| > {EXTREME_DELAY_MIN} min")
print("Extreme rows:", share(int(extreme.sum()), int(delay.notna().sum())))
print(f"  Late  (above +{EXTREME_DELAY_MIN}): {int(too_late.sum()):,}")
print(f"  Early (below -{EXTREME_DELAY_MIN}): {int(too_early.sum()):,}")
print("  Also in a ride-key duplicate group:", share(int((extreme & in_ride_group).sum()), int(extreme.sum())))

print("\nExtreme rows by train type:")
display(df.loc[extreme, "train_type"].value_counts().head(10))

Threshold (proposal): |delay| > 180 min
Extreme rows: 1,372 of 14,276,283 (0.0%)
  Late  (above +180): 1,371
  Early (below -180): 1
  Also in a ride-key duplicate group: 1,207 of 1,372 (88.0%)

Extreme rows by train type:


train_type
ICE    326
Bus    292
RE     114
RB     101
NJ      98
RJ      65
ARV     58
FLX     57
IC      52
AVG     22
Name: count, dtype: int64

## 6 Planned hours per service and station

Hypothesis: a recurring service passes a station at about the same hour every day, so "hour" adds little once service x station is known. Service = train type + train number (the board said train number only; the type is added so that, for example, S 1 and RE 1 are never merged). A train planned at 8:59 on some days and 9:01 on others counts as two hours, so a value of 2 is not always a real schedule change.

In [26]:
planned = df["departure_planned_time"].fillna(df["arrival_planned_time"])
hours = planned.dt.hour.rename("planned_hour")
pair_keys = [df[column] for column in [*SERVICE_KEY, "eva"]]
pairs = hours.groupby(pair_keys, observed=True).agg(["size", "nunique"])

one_hour = pairs["nunique"] == 1
print(f"Service x station pairs: {len(pairs):,}")
print("  With exactly one planned hour (share of pairs):", share(int(one_hour.sum()), len(pairs)))
print("  Stops in those pairs (share of stops):         ",
      share(int(pairs.loc[one_hour, "size"].sum()), int(pairs["size"].sum())))

print("\nPairs by number of distinct planned hours:")
display(pairs["nunique"].value_counts().sort_index().head(10))

Service x station pairs: 851,617
  With exactly one planned hour (share of pairs): 828,371 of 851,617 (97.3%)
  Stops in those pairs (share of stops):          13,910,115 of 14,276,283 (97.4%)

Pairs by number of distinct planned hours:


nunique
1     828371
2      20501
3       1446
4        430
5        302
6        188
7        141
8        106
9         56
10        43
Name: count, dtype: int64

## 3f Cancellations vs planned departure

Rows with no planned departure leave P(late). If cancelled stops often have no planned departure, removing those rows from a shared table would also remove most cancellations, and P(cancelled) would need its own base table. This cell checks how much the two overlap.

In [27]:
cancelled = df["arrival_is_canceled"] | df["departure_is_canceled"]
n_cancelled = int(cancelled.sum())
print("Cancelled stops (either flag):", share(n_cancelled, len(df)))
print("  ...with no planned departure:", share(int((cancelled & no_departure).sum()), n_cancelled))

print("\nEither flag vs no planned departure:")
print(pd.crosstab(no_departure.rename("no planned departure"),
                  cancelled.rename("cancelled (either flag)"), margins=True))

print("\nDeparture flag only vs no planned departure:")
print(pd.crosstab(no_departure.rename("no planned departure"),
                  df["departure_is_canceled"].rename("departure cancelled"), margins=True))

Cancelled stops (either flag): 651,697 of 14,768,104 (4.4%)
  ...with no planned departure: 59,349 of 651,697 (9.1%)

Either flag vs no planned departure:
cancelled (either flag)     False    True       All
no planned departure                               
False                    12557421  592348  13149769
True                      1558986   59349   1618335
All                      14116407  651697  14768104

Departure flag only vs no planned departure:
departure cancelled      False    True       All
no planned departure                            
False                 12580489  569280  13149769
True                   1614283    4052   1618335
All                   14194772  573332  14768104


## 12 Cancel switches (stops with a planned departure)

A stop has two cancel switches. If only the ARRIVAL is cancelled, the train may still leave from here (for example, a run that now starts at this station). The table shows all four switch combinations. The delay count is a formality (3a: every stop with a planned departure has a delay). The real test is the last two columns: if arrival-only stops run late about as often as stops with no cancellation, they probably ran. That is a hypothesis to check, not proof.

In [28]:
with_departure = df.loc[~no_departure, [DELAY_COLUMN, *CANCEL_COLUMNS]]
is_late = with_departure[DELAY_COLUMN] >= LATE_THRESHOLD_MIN

switches = with_departure.groupby(CANCEL_COLUMNS).agg(
    stops=(DELAY_COLUMN, "size"),
    with_delay=(DELAY_COLUMN, "count"),
    median_delay=(DELAY_COLUMN, "median"),
)
switches["late_share"] = is_late.groupby([with_departure[c] for c in CANCEL_COLUMNS]).mean().round(3)
print("Stops with a planned departure:", f"{len(with_departure):,}")
display(switches)

arrival_only = with_departure["arrival_is_canceled"] & ~with_departure["departure_is_canceled"]
print("Arrival-only cancellations:", share(int(arrival_only.sum()), len(with_departure)))

Stops with a planned departure: 13,149,769


stops  with_delay  median_delay  \
arrival_is_canceled departure_is_canceled                                       
False               False                  12557421    12557421           1.0   
                    True                      77159       77159           0.0   
True                False                     23068       23068           1.0   
                    True                     492121      492121           0.0   

                                           late_share  
arrival_is_canceled departure_is_canceled              
False               False                       0.160  
                    True                        0.228  
True                False                       0.232  
                    True                        0.168

Arrival-only cancellations: 23,068 of 13,149,769 (0.2%)


## 13 Month boundaries (all six files, key and planned-time columns only)

The July file holds stops planned on June 30, so files seem to be filed by actual time, not planned time. Two questions per file: how many stops are planned outside the file's month, and how many stops with a planned departure also appear in the previous file (same timetable key). A stop in two files would be counted twice in monthly totals, and could leak August stops into September.

To save memory, each stop's key is turned into a "fingerprint" (one number per stop, like a barcode), and only the previous month's fingerprints are kept while the next file loads. The large July tables from the cells above are deleted first.

Two controls make sure a zero overlap is real and not a bug: July's extra copies within the file must equal 3b's 1,255, and the two column types must be the same in all six rows.

In [29]:
del df, pair_keys, delay, with_departure   # free July's tables before reading six files one at a time

POSITIONS = ["earlier", "this month", "later", "missing"]
boundary_rows = []
previous_keys = None
for month in MONTHS:
    stops = pd.read_parquet(month_path(month), columns=BOUNDARY_COLUMNS)
    position_counts = month_position(planned_day(stops), month).value_counts()
    keys = key_fingerprints(stops)
    unique_keys = np.unique(keys)
    in_previous = None if previous_keys is None else int(np.isin(keys, previous_keys).sum())
    boundary_rows.append({
        "month": month,
        "rows": len(stops),
        **{position: int(position_counts.get(position, 0)) for position in POSITIONS},
        "with_departure": len(keys),
        "extra_copies_within_file": len(keys) - len(unique_keys),   # control: July must match 3b
        "key_also_in_previous_file": in_previous,
        # Fingerprints only match across files if the columns have the same type everywhere.
        "eva_type": str(stops["eva"].dtype),
        "departure_time_type": str(stops["departure_planned_time"].dtype),
    })
    previous_keys = unique_keys
    del stops

boundaries = pd.DataFrame(boundary_rows).set_index("month")
display(boundaries)

,rows,earlier,this month,later,missing,with_departure,extra_copies_within_file,key_also_in_previous_file,eva_type,departure_time_type
month,,,,,,,,,,
2026-04,14555946,631,14294266,0,261049,13201847,273,NaN,str,datetime64[ns]
2026-05,14906549,788,14604357,0,301404,13484454,1165,0.0,str,datetime64[ns]
2026-06,15082144,817,15003579,0,77748,13838849,492,0.0,str,datetime64[ns]
2026-07,14768104,1162,14275116,5,491821,13149769,1255,0.0,str,datetime64[ns]
2026-08,14516986,1142,14479198,6,36640,13305252,633,0.0,str,datetime64[ns]
2026-09,14765419,811,14764600,3,5,13626593,327,0.0,str,datetime64[ns]


## Not in this notebook

3d (buses) is no longer needed: the owner decided to drop train type Bus and keep SBH.

Task 13 frees July's table (`del df`), so to rerun an earlier cell, use Restart and Run All.